# EEEM030 Group Assignment 2: Speech Recognition

Isolated-word recognition (11 words) with an FFN and a Transformer.

**Team members / chapter authors:** _TBC_

## 0. Setup

### 0.1 Imports

In [ ]:
from pathlib import Path
from collections import Counter
from io import BytesIO
import hashlib
import json
import math
import platform
import time
import urllib.request
import zipfile

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import soundfile as sf
import torch
import torchaudio
from torch import nn
from IPython.display import Audio, Markdown, display


def parse_recording(path):
    speaker, recording, word = path.stem.split('_')

    return {
        'path': path,
        'speaker': speaker,
        'recording': int(recording[1:]),
        'word': word
    }

paths = sorted(Path("DevelopmentSet").glob("*.mp3"))

records = [parse_recording(path) for path in paths]

# Speakers list
all_speakers = sorted({r['speaker'] for r in records})

# Words list
all_words = sorted({r['word'] for r in records})

print("Speakers:", all_speakers)
print("Words:", all_words)

# Nbr of recordings
print("Total recordings:", len(records))

counts = Counter((r['speaker'], r['word']) for r in records)

print("Recordings per speaker:", set(counts.values()))

Speakers : ['sp01a', 'sp02a', 'sp03a', 'sp04a', 'sp05a', 'sp06a', 'sp07a', 'sp08a', 'sp09a', 'sp10a', 'sp11a', 'sp12a', 'sp13a', 'sp14a', 'sp15a', 'sp16a', 'sp17a', 'sp18a', 'sp19a', 'sp20a', 'sp21a', 'sp22a', 'sp23a', 'sp24a', 'sp25a', 'sp26a', 'sp27a', 'sp28a', 'sp29a', 'sp30a']
Words : ['had', 'hard', 'head', 'heard', 'heed', 'hid', 'hoard', 'hod', 'hood', 'hud', 'whod']
Total recordings : 330
Recordings per speaker/word : {1}


Training, validation and test set

In [18]:
TRAIN_SPEAKERS = {'sp01a', 'sp02a', 'sp03a', 'sp04a', 'sp05a', 'sp06a', 'sp07a', 'sp08a', 'sp09a', 'sp10a',
                  'sp11a', 'sp12a', 'sp13a', 'sp14a', 'sp15a', 'sp16a', 'sp17a', 'sp18a', 'sp19a', 'sp20a',
                  'sp21a', 'sp22a', 'sp23a', 'sp24a', 'sp25a', 'sp26a', 'sp27a', 'sp28a'}
VAL_SPEAKERS = {'sp29a'}
TEST_SPEAKERS = {'sp30a'}

# Training speakers must not appear in validation or test.
assert TRAIN_SPEAKERS.isdisjoint(VAL_SPEAKERS | TEST_SPEAKERS)

# Validation and test speakers must not overlap.
assert VAL_SPEAKERS.isdisjoint(TEST_SPEAKERS)

# The three splits must collectively contain exactly the dataset's speakers.
assert TRAIN_SPEAKERS | VAL_SPEAKERS | TEST_SPEAKERS == set(all_speakers)

split_speakers = {'train': TRAIN_SPEAKERS, 'validation': VAL_SPEAKERS, 'test': TEST_SPEAKERS}
split_indices = {
    split: torch.tensor([i for i, r in enumerate(records) if r['speaker'] in speakers],
                        dtype=torch.long)
    for split, speakers in split_speakers.items()
}
for split, indices in split_indices.items():
    labels = Counter(records[i]['word'] for i in indices.tolist())
    print(f'{split:<11}: {len(indices):4d} recordings | per-word counts: {dict(sorted(labels.items()))}')
    # Check that the split contains all eleven word labels with no other labels.
    assert set(labels) == set(all_words)


train      :  308 recordings | per-word counts: {'had': 28, 'hard': 28, 'head': 28, 'heard': 28, 'heed': 28, 'hid': 28, 'hoard': 28, 'hod': 28, 'hood': 28, 'hud': 28, 'whod': 28}
validation :   11 recordings | per-word counts: {'had': 1, 'hard': 1, 'head': 1, 'heard': 1, 'heed': 1, 'hid': 1, 'hoard': 1, 'hod': 1, 'hood': 1, 'hud': 1, 'whod': 1}
test       :   11 recordings | per-word counts: {'had': 1, 'hard': 1, 'head': 1, 'heard': 1, 'heed': 1, 'hid': 1, 'hoard': 1, 'hod': 1, 'hood': 1, 'hud': 1, 'whod': 1}


### 0.2 Environment specification & random seeds

### 0.3 Global hyper-parameters / config

## Part 2: Data Preparation & Feed-Forward Network

### 2.1 Load speech recordings & parse labels

### 2.2 Log-mel spectrogram extraction (80 mel bins, 25 ms window, 10 ms hop)

### 2.3 Fixed-length representation (pooling / pad-truncate / sequence)

### 2.4 Train / validation / test split (speaker-independent)

### 2.5 Plot example mel-spectrograms with labels

### 2.6 MLP from scratch (NumPy): forward pass

### 2.7 MLP in PyTorch (`nn.Module`)

### 2.8 Worked numerical check: NumPy vs PyTorch outputs

### 2.9 Shape diagram / dimension counting

## Part 3: Training with Backpropagation & SGD

### 3.1 NumPy MLP: backpropagation by hand

#### 3.1.1 Gradient derivation (cross-entropy + softmax, chain rule)

#### 3.1.2 Backward pass implementation

#### 3.1.3 Vanilla SGD / mini-batch training loop

#### 3.1.4 Loss & accuracy tracking (train / validation)

### 3.2 PyTorch MLP training

#### 3.2.1 Training with `torch.optim.SGD`

#### 3.2.2 Training with Adam

#### 3.2.3 Regularisation: weight decay & dropout

#### 3.2.4 Augmentation: SpecAugment / additive noise

### 3.3 Learning-rate schedules & early stopping

### 3.4 Results: loss & accuracy curves

### 3.5 Comparison table (accuracy, confusion matrix, parameters, training time)

### 3.6 Discussion: effect of dropout / SpecAugment on train–validation gap

## Part 4: Attention Mechanism

### 4.1 Scaled dot-product attention (from scratch)

### 4.2 Multi-head attention

### 4.3 Worked example: 4–8 frame toy sequence (Q, K, V, weights, output)

### 4.4 Verification against implementation

### 4.5 Self-attention on mel-spectrogram frames

### 4.6 Attention weight visualisations & interpretation

## Part 5: Basic Transformer Encoder for Mel-Spectrograms

### 5.1 Positional encodings (sinusoidal / learned)

### 5.2 Convolutional subsampling front-end (optional)

### 5.3 Transformer encoder block (MHSA + residual + LayerNorm, FFN + residual + LayerNorm)

### 5.4 Full model: 2 stacked encoder layers + classification head

### 5.5 Training & inference methods

### 5.6 Training curves & final test accuracy

### 5.7 Comparison vs MLP (accuracy & parameter count)

### 5.8 Analysis: does attention help? Effect of sequence length after subsampling

## Final Evaluation on Test Data

## Discussion Questions

### Why is the data-driven approach particularly powerful for speech?

### How does the √d_k scaling prevent vanishing gradients in the softmax?

### FFN on pooled features vs Transformer on full sequence: trade-offs

## Stretch Objectives (optional)

## References & Third-Party Code Attribution